# Top-p (nucleus) sweep: BLEU-4 as a function of `top_p`

Loads one trained checkpoint and decodes the eval images with **nucleus sampling** at
several `top_p` values, reporting corpus / mean BLEU-4 for each. A deterministic **greedy
baseline** is drawn for reference (greedy == `top_p -> 0`).

- Nucleus sampling is **stochastic**, so each `top_p` is averaged over `NUM_SAMPLES_PER_P`
  draws (mean +/- std on the plot).
- Expect BLEU to *fall* as `top_p` rises (more diversity, less reference overlap) - the
  point is to see the fluency/diversity vs. BLEU trade-off, not to beat greedy.
- Works with any checkpoint the training notebooks saved (CNN/ViT/CLIP + GPT-2, or CNN+GRU).

## 1. Install dependencies and imports

In [ ]:
# Install once if needed
!pip -q install transformers pycocotools nltk

import os, json, random, time, copy, math, textwrap, shutil
from dataclasses import dataclass, asdict, fields
from typing import Optional
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision import models

from transformers import (
    AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer,
    AutoImageProcessor, CLIPVisionModel,
)

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Using device:", device)

## 2. Settings & mount Drive

Point `MODEL_PATH` at your checkpoint and choose the `TOP_P_VALUES` to sweep. Lower
`top_p`/`temperature` -> safer, closer to greedy; higher -> more diverse.

In [ ]:
# Portable paths: works on Colab (mounts Drive) AND a local Jupyter notebook.
try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content"
    ON_COLAB = True
except ImportError:
    BASE_DIR = os.path.abspath(".")
    ON_COLAB = False

DATA_DIR = os.path.join(BASE_DIR, "data", "coco")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs_topp")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- The checkpoint to evaluate (edit this) ---------------------------------
MODEL_PATH = "/content/drive/MyDrive/image_captioning_finetune/vit_gpt2_best.pt"

# ---- What to evaluate on -----------------------------------------------------
COCO_SPLIT = "val"        # which COCO image set is on disk
EVAL_SPLIT = "val"        # "val" = clean held-out 10% (recommended); "all" = every image
NUM_EVAL_IMAGES = 300     # e.g. 300 for a quick pass; None = all in EVAL_SPLIT

# ---- Top-p (nucleus) sweep ---------------------------------------------------
TOP_P_VALUES = [0.5, 0.7, 0.8, 0.9, 0.95, 1.0]   # nucleus mass kept at each setting
TEMPERATURE = 1.0          # held fixed across the sweep; >1 flatter, <1 sharper
NUM_SAMPLES_PER_P = 2      # stochastic draws per image per p (averaged to reduce noise)
MAX_GEN_LEN = 40
MAX_TEXT_LEN = 40

print("Model path:", MODEL_PATH)
print("Output dir:", OUTPUT_DIR)
print("top_p values:", TOP_P_VALUES, "| T:", TEMPERATURE, "| samples/p:", NUM_SAMPLES_PER_P)
print("eval split:", EVAL_SPLIT, "| cap:", NUM_EVAL_IMAGES)

## 3. Download MS-COCO data

In [ ]:
import urllib.request, zipfile

ANNOTATIONS_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
IMAGES_URL = ("http://images.cocodataset.org/zips/val2017.zip" if COCO_SPLIT == "val"
              else "http://images.cocodataset.org/zips/train2017.zip")

ann_dir = os.path.join(DATA_DIR, "annotations")
img_dir = os.path.join(DATA_DIR, f"{COCO_SPLIT}2017")

def _download_and_extract(url, zip_path, extract_to):
    def _progress(block_num, block_size, total_size):
        if total_size > 0:
            pct = min(100, block_num * block_size * 100 / total_size)
            print(f"\r  downloading... {pct:5.1f}%", end="")
    print("Downloading", url)
    urllib.request.urlretrieve(url, zip_path, _progress)
    print("\n  extracting...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    os.remove(zip_path)
    print("  done.")

if not os.path.exists(ann_dir):
    _download_and_extract(ANNOTATIONS_URL, os.path.join(DATA_DIR, "annotations.zip"), DATA_DIR)
else:
    print("Annotations already extracted.")

if not os.path.exists(img_dir):
    _download_and_extract(IMAGES_URL, os.path.join(DATA_DIR, f"{COCO_SPLIT}2017.zip"), DATA_DIR)
else:
    print(f"{COCO_SPLIT}2017 images already extracted.")

## 4. Load captions and reproduce the train/val split

In [ ]:
annotations_file = os.path.join(DATA_DIR, "annotations", f"captions_{COCO_SPLIT}2017.json")
with open(annotations_file, "r") as f:
    coco_data = json.load(f)

img_id_to_filename = {img["id"]: img["file_name"] for img in coco_data["images"]}
img_id_to_captions = defaultdict(list)
for ann in coco_data["annotations"]:
    img_id_to_captions[ann["image_id"]].append(ann["caption"])

def make_image_id_split(image_ids, train_fraction=0.90, seed=42):
    image_ids = list(image_ids)
    rng = random.Random(seed)
    rng.shuffle(image_ids)
    split_idx = int(train_fraction * len(image_ids))
    return set(image_ids[:split_idx]), set(image_ids[split_idx:])

train_img_ids, val_img_ids = make_image_id_split(img_id_to_filename.keys(), 0.90, SEED)
train_annotations = [a for a in coco_data["annotations"] if a["image_id"] in train_img_ids]

print("Total images:", len(img_id_to_filename))
print("Train images:", len(train_img_ids), " Val images:", len(val_img_ids))

## 5. Unified encoder/decoder framework

In [ ]:
# ---- Word-level vocabulary (for GRU checkpoints) ----------------------------
class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.freq_threshold = freq_threshold
        self.word2idx = {"<pad>": 0, "<start>": 1, "<end>": 2, "<unk>": 3}
        self.idx2word = {0: "<pad>", 1: "<start>", 2: "<end>", 3: "<unk>"}
        self.word_count = Counter()

    def build(self, captions):
        for cap in captions:
            self.word_count.update(word_tokenize(cap.lower()))
        idx = 4
        for w, c in self.word_count.items():
            if c >= self.freq_threshold:
                self.word2idx[w] = idx; self.idx2word[idx] = w; idx += 1

    def decode(self, ids):
        words = []
        for i in ids:
            w = self.idx2word.get(int(i), "<unk>")
            if w == "<end>": break
            if w not in ("<start>", "<pad>"): words.append(w)
        return " ".join(words)

    def __len__(self):
        return len(self.word2idx)

rnn_vocab = Vocabulary(freq_threshold=5)
rnn_vocab.build([a["caption"] for a in train_annotations])
print("GRU word-level vocab size:", len(rnn_vocab))

# ---- Image preprocessing ----------------------------------------------------
resnet_transform = T.Compose([
    T.Resize((256, 256)), T.CenterCrop(224), T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])

def preprocess_images(images, encoder_kind, image_processor):
    if encoder_kind == "cnn":
        return torch.stack([resnet_transform(im) for im in images], 0)
    return image_processor(list(images), return_tensors="pt").pixel_values

# ---- Encoder: image -> sequence of feature vectors (B, S, D_enc) ------------
class ImageEncoder(nn.Module):
    def __init__(self, kind, name):
        super().__init__()
        self.kind = kind
        if kind == "cnn":
            resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.backbone = nn.Sequential(*list(resnet.children())[:-2])
            self.feat_dim = 2048
        elif kind == "clip":
            self.backbone = CLIPVisionModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size
        else:  # vit
            self.backbone = AutoModel.from_pretrained(name)
            self.feat_dim = self.backbone.config.hidden_size

    def forward(self, images):
        if self.kind == "cnn":
            f = self.backbone(images)
            B, C, H, W = f.shape
            return f.view(B, C, H * W).permute(0, 2, 1)
        out = self.backbone(pixel_values=images)
        return out.last_hidden_state

# ---- Decoder A: word-level GRU ----------------------------------------------
class GRUDecoder(nn.Module):
    def __init__(self, feat_dim, vocab, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.vocab = vocab
        self.pad_id = vocab.word2idx["<pad>"]
        self.end_id = vocab.word2idx["<end>"]
        self.img_proj = nn.Linear(feat_dim, embed_size)
        self.bn = nn.BatchNorm1d(embed_size)
        self.embed = nn.Embedding(len(vocab), embed_size)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, len(vocab))

    def _img_token(self, enc_seq):
        pooled = enc_seq.mean(dim=1)
        return self.bn(self.img_proj(pooled))

    def decode(self, ids):
        return self.vocab.decode(ids)

# ---- Decoder B: GPT-2 with cross-attention ----------------------------------
class GPT2Decoder(nn.Module):
    def __init__(self, feat_dim, tokenizer, dropout=0.1, freeze_base=False):
        super().__init__()
        cfg = AutoConfig.from_pretrained("gpt2")
        cfg.is_decoder = True
        cfg.add_cross_attention = True
        cfg.resid_pdrop = dropout
        cfg.embd_pdrop = dropout
        cfg.attn_pdrop = dropout
        self.gpt2 = AutoModelForCausalLM.from_pretrained("gpt2", config=cfg)
        self.gpt2.resize_token_embeddings(len(tokenizer))
        self.enc_proj = nn.Linear(feat_dim, cfg.n_embd)
        self.tokenizer = tokenizer
        self.bos_id = tokenizer.bos_token_id
        self.eos_id = tokenizer.eos_token_id
        self.pad_id = tokenizer.pad_token_id

    def decode(self, ids):
        return self.tokenizer.decode(ids, skip_special_tokens=True).strip()

# ---- Full model -------------------------------------------------------------
class CaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.freeze_encoder = freeze_encoder

    @torch.no_grad()
    def encode(self, images):
        return self.encoder(images)

    def decode(self, ids):
        return self.decoder.decode(ids)

# ---- Config + builder -------------------------------------------------------
@dataclass
class ExperimentConfig:
    name: str
    encoder_kind: str
    encoder_name: str
    decoder: str
    learning_rate: float
    weight_decay: float = 0.0
    dropout: float = 0.1
    freeze_gpt2_base: bool = False
    embed_size: int = 256
    hidden_size: int = 512
    num_layers: int = 2
    freq_threshold: int = 5
    batch_size: int = 16
    epochs: int = 15
    max_train_batches: Optional[int] = 300

def build_model(config):
    encoder = ImageEncoder(config.encoder_kind, config.encoder_name)
    feat_dim = encoder.feat_dim
    if config.decoder == "gru":
        decoder = GRUDecoder(feat_dim, rnn_vocab, config.embed_size,
                             config.hidden_size, config.num_layers, config.dropout)
    elif config.decoder == "gpt2":
        tok = AutoTokenizer.from_pretrained("gpt2")
        if tok.pad_token is None:
            tok.add_special_tokens({"pad_token": "<|pad|>"})
        decoder = GPT2Decoder(feat_dim, tok, dropout=config.dropout,
                              freeze_base=config.freeze_gpt2_base)
    else:
        raise ValueError(f"Unknown decoder: {config.decoder}")
    model = CaptioningModel(encoder, decoder, freeze_encoder=True)
    image_processor = (None if config.encoder_kind == "cnn"
                       else AutoImageProcessor.from_pretrained(config.encoder_name))
    return {"model": model, "encoder_kind": config.encoder_kind,
            "decoder_kind": config.decoder, "image_processor": image_processor}

print("Framework ready.")

## 6. Load the checkpoint

In [ ]:
ckpt = torch.load(MODEL_PATH, map_location=device)
# Some checkpoints (e.g. the ViT-only notebook) don't store every field this
# notebook's ExperimentConfig expects. Infer/fill the missing ones.
saved_cfg = dict(ckpt["config"])
if "encoder_kind" not in saved_cfg:
    _nm = str(saved_cfg.get("encoder_name", "")).lower()
    if "clip" in _nm:
        saved_cfg["encoder_kind"] = "clip"
    elif "resnet" in _nm or _nm == "cnn":
        saved_cfg["encoder_kind"] = "cnn"
    else:
        saved_cfg["encoder_kind"] = "vit"
saved_cfg.setdefault("name", os.path.splitext(os.path.basename(MODEL_PATH))[0])
saved_cfg.setdefault("decoder", "gpt2")

valid = {f.name for f in fields(ExperimentConfig)}
cfg_dict = {k: v for k, v in saved_cfg.items() if k in valid}
config = ExperimentConfig(**cfg_dict)
MODEL_NAME = config.name

bundle = build_model(config)
model = bundle["model"].to(device)
missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
model.eval()

print("Loaded:", MODEL_NAME)
print(f"  encoder = {config.encoder_kind} ({config.encoder_name})")
print(f"  decoder = {config.decoder}")
print(f"  reported val BLEU-4 at save time = {ckpt.get('bleu4')}  (greedy, epoch {ckpt.get('epoch')})")
if missing:    print("  [warn] missing keys:", len(missing))
if unexpected: print("  [warn] unexpected keys:", len(unexpected))

## 7. Nucleus (top-p) sampling decoders

At each step: temperature-scale the logits, softmax, keep the smallest set of tokens whose
cumulative probability reaches `top_p`, renormalise, and **sample** one. `top_p -> 0`
collapses to greedy (argmax).

In [ ]:
@torch.no_grad()
def _nucleus_pick(logits, top_p, temperature):
    """Sample one token id from the top-p (nucleus) of a 1-D logits vector.
    top_p <= 0 reduces to greedy argmax."""
    logits = logits / max(temperature, 1e-6)
    probs = F.softmax(logits, dim=-1)
    sp, si = torch.sort(probs, descending=True)
    cdf = torch.cumsum(sp, dim=-1)
    keep = cdf <= top_p
    keep[0] = True                       # always keep the most likely token
    sp, si = sp[keep], si[keep]
    nxt = si[torch.multinomial(sp / sp.sum(), 1)]
    return int(nxt.item())


@torch.no_grad()
def nucleus_sample_gpt2(decoder, enc_seq_1, max_len, top_p=0.9, temperature=1.0):
    dev = enc_seq_1.device
    enc_hidden = decoder.enc_proj(enc_seq_1)
    bos, eos = decoder.bos_id, decoder.eos_id
    seq = [bos]
    for _ in range(max_len):
        ids = torch.tensor([seq], device=dev)
        out = decoder.gpt2(input_ids=ids, encoder_hidden_states=enc_hidden)
        nxt = _nucleus_pick(out.logits[:, -1, :].squeeze(0), top_p, temperature)
        if nxt == eos:
            break
        seq.append(nxt)
    return [t for t in seq if t not in (bos, eos)]


@torch.no_grad()
def nucleus_sample_gru(decoder, enc_seq_1, max_len, top_p=0.9, temperature=1.0):
    dev = enc_seq_1.device
    end = decoder.end_id
    feat = decoder._img_token(enc_seq_1)
    out, states = decoder.rnn(feat.unsqueeze(1))         # image -> first token
    nxt = _nucleus_pick(decoder.linear(out.squeeze(1)).squeeze(0), top_p, temperature)
    seq = []
    for _ in range(max_len):
        if nxt == end:
            break
        seq.append(nxt)
        emb = decoder.embed(torch.tensor([nxt], device=dev)).unsqueeze(1)
        out, states = decoder.rnn(emb, states)
        nxt = _nucleus_pick(decoder.linear(out.squeeze(1)).squeeze(0), top_p, temperature)
    return seq


@torch.no_grad()
def caption_nucleus(model, pixel_values_1, max_len, top_p=0.9, temperature=1.0):
    enc_seq = model.encode(pixel_values_1)
    dec = model.decoder
    if isinstance(dec, GPT2Decoder):
        ids = nucleus_sample_gpt2(dec, enc_seq, max_len, top_p, temperature)
    else:
        ids = nucleus_sample_gru(dec, enc_seq, max_len, top_p, temperature)
    return model.decode(ids)

print("Nucleus sampling ready. (top_p -> 0 behaves like greedy)")

## 8. Sweep `top_p` and score BLEU-4

For each `top_p`, every eval image is decoded `NUM_SAMPLES_PER_P` times. We report the
**corpus BLEU-4** averaged over draws (with std), the **mean per-image BLEU-4** over all
draws, and the average caption length. The RNG is reseeded per `top_p` so the draws are
comparable across settings.

In [ ]:
if EVAL_SPLIT == "val":
    eval_ids = [i for i in img_id_to_filename if i in val_img_ids]
else:
    eval_ids = list(img_id_to_filename.keys())
eval_ids = sorted(eval_ids)
if NUM_EVAL_IMAGES is not None:
    eval_ids = eval_ids[:NUM_EVAL_IMAGES]
print("Sweeping top-p on", len(eval_ids), "images |", TOP_P_VALUES,
      "| samples/p", NUM_SAMPLES_PER_P, "| T", TEMPERATURE)

kind = bundle["encoder_kind"]
image_processor = bundle["image_processor"]
smoothing = SmoothingFunction().method1
W4 = (0.25, 0.25, 0.25, 0.25)
ref_tok = {iid: [word_tokenize(c.lower()) for c in img_id_to_captions[iid]] for iid in eval_ids}
refs = [ref_tok[iid] for iid in eval_ids]

def load_pixel(iid):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[iid])).convert("RGB")
    return preprocess_images([img], kind, image_processor).to(device)

# --- deterministic greedy baseline (top_p -> 0) for reference ---------------
t0 = time.time()
greedy_hyps = []
for iid in eval_ids:
    cap = caption_nucleus(model, load_pixel(iid), MAX_GEN_LEN, top_p=0.0, temperature=1.0)
    greedy_hyps.append(word_tokenize(cap.lower()))
greedy_corpus = corpus_bleu(refs, greedy_hyps, smoothing_function=smoothing)
greedy_mean = float(np.mean([sentence_bleu(r, h, weights=W4, smoothing_function=smoothing)
                             if h else 0.0 for r, h in zip(refs, greedy_hyps)]))
print(f"greedy baseline: corpus {greedy_corpus:.4f} | mean {greedy_mean:.4f} "
      f"({time.time()-t0:.0f}s)")

# --- top-p sweep -----------------------------------------------------------
results = []
per_p_caps = {}   # per_p_caps[p] = {iid: first-draw caption}  (for qualitative cell)
for p in TOP_P_VALUES:
    torch.manual_seed(SEED)        # same RNG start for every p -> fair comparison
    corpus_per_draw, all_sent, all_len = [], [], []
    first_caps = {}
    t0 = time.time()
    for s in range(NUM_SAMPLES_PER_P):
        hyps = []
        for iid in eval_ids:
            cap = caption_nucleus(model, load_pixel(iid), MAX_GEN_LEN, p, TEMPERATURE)
            if s == 0:
                first_caps[iid] = cap
            toks = word_tokenize(cap.lower())
            hyps.append(toks)
            all_len.append(len(toks))
            all_sent.append(sentence_bleu(ref_tok[iid], toks, weights=W4,
                                          smoothing_function=smoothing) if toks else 0.0)
        corpus_per_draw.append(corpus_bleu(refs, hyps, smoothing_function=smoothing))
    results.append({
        "top_p": p,
        "corpus_bleu4": float(np.mean(corpus_per_draw)),
        "corpus_bleu4_std": float(np.std(corpus_per_draw)),
        "mean_bleu4": float(np.mean(all_sent)),
        "avg_caption_len": float(np.mean(all_len)),
        "seconds": round(time.time() - t0, 1),
    })
    per_p_caps[p] = first_caps
    r = results[-1]
    print(f"  p={p}: corpus {r['corpus_bleu4']:.4f} (+/-{r['corpus_bleu4_std']:.4f}) | "
          f"mean {r['mean_bleu4']:.4f} | len {r['avg_caption_len']:.1f} | {r['seconds']:.0f}s")

res_df = pd.DataFrame(results)
res_csv = os.path.join(OUTPUT_DIR, f"top_p_comparison_{MODEL_NAME}.csv")
res_df.to_csv(res_csv, index=False)
print("\nsaved:", res_csv)
res_df

## 9. Plot - BLEU-4 vs `top_p`

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(res_df["top_p"], res_df["corpus_bleu4"], yerr=res_df["corpus_bleu4_std"],
            marker="o", linewidth=2, capsize=4, label="nucleus corpus BLEU-4")
ax.plot(res_df["top_p"], res_df["mean_bleu4"], marker="s", linewidth=2,
        label="mean per-image BLEU-4")
ax.axhline(greedy_corpus, color="gray", ls="--",
           label=f"greedy corpus BLEU-4 ({greedy_corpus:.3f})")
ax.set_xlabel("top_p  (nucleus mass kept)")
ax.set_ylabel("BLEU-4")
ax.set_title(f"BLEU-4 vs top_p  -  {MODEL_NAME}  (T={TEMPERATURE})")
ax.set_xticks(TOP_P_VALUES)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
sweep_png = os.path.join(OUTPUT_DIR, f"top_p_bleu_{MODEL_NAME}.png")
fig.savefig(sweep_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", sweep_png)

best_row = res_df.loc[res_df["corpus_bleu4"].idxmax()]
print(f"Best nucleus corpus BLEU-4 at top_p={best_row['top_p']} "
      f"({best_row['corpus_bleu4']:.4f}); greedy baseline = {greedy_corpus:.4f}.")

## 10. Qualitative - how a caption changes with `top_p`

In [ ]:
sample_ids = eval_ids[:4]
fig, axes = plt.subplots(1, len(sample_ids), figsize=(len(sample_ids) * 4.6, 5.8))
axes = np.array(axes).reshape(-1)
for ax, iid in zip(axes, sample_ids):
    img = Image.open(os.path.join(img_dir, img_id_to_filename[iid])).convert("RGB")
    ax.imshow(img); ax.axis("off")
    lines = [f"p{p}: {per_p_caps[p][iid]}" for p in TOP_P_VALUES]
    ax.set_title("\n".join(textwrap.fill(l, 36) for l in lines), fontsize=7)
fig.suptitle(f"Caption vs top_p (one draw each)  -  {MODEL_NAME}", fontsize=13)
fig.tight_layout()
ex_png = os.path.join(OUTPUT_DIR, f"top_p_examples_{MODEL_NAME}.png")
fig.savefig(ex_png, dpi=150, bbox_inches="tight")
plt.show()
print("saved:", ex_png)

## 11. Save outputs to Drive (optional)

In [ ]:
TOPP_DRIVE_DIR = "/content/drive/MyDrive/image_captioning_topp"
out_files = [res_csv, sweep_png, ex_png]
if ON_COLAB:
    os.makedirs(TOPP_DRIVE_DIR, exist_ok=True)
    for f in out_files:
        if os.path.exists(f):
            shutil.copy2(f, os.path.join(TOPP_DRIVE_DIR, os.path.basename(f)))
    print("Copied top-p outputs to Drive:", TOPP_DRIVE_DIR)
else:
    print("Not on Colab - outputs persist locally under:", os.path.abspath(OUTPUT_DIR))